# Vertex AI AutoML Tabular Inference (Online & Batch)

This tutorial demonstrates model serving and inference options for AutoML Tabular models using `tabflows`.

### Objectives
1. Discover trained AutoML Tabular `aiplatform.Model` resources from Vertex AI Model Registry or a completed `PipelineJob`.
2. Deploy the model to a real-time `aiplatform.Endpoint` and execute Online Inference.
3. Submit a Batch Prediction job against a GCS dataset and retrieve output predictions.
4. Clean up deployed endpoint resources to manage serving costs.

In [ ]:
import os

from dotenv import load_dotenv
from google.cloud import aiplatform

from tabflows import (
    TabularPipelineConfig,
    list_models,
)

# Load environment variables from local .env file
load_dotenv()
print("Environment and libraries loaded successfully.")

In [ ]:
# TabularPipelineConfig automatically loads GCP_PROJECT, GCP_LOCATION, GCP_BUCKET_URI
config = TabularPipelineConfig()

print(f"Project ID: {config.project_id}")
print(f"Location: {config.location}")
print(f"Serving Machine Type: {config.serving_machine_type}")

# Option 1: Discover recent models automatically from Vertex AI Model Registry
print("\n--- Discovering Recent Models in Vertex AI ---")
try:
    recent_models = list_models(config=config, limit=5)
    if recent_models:
        print(f"Found {len(recent_models)} recent model(s):")
        for m in recent_models:
            print(f"  - Name: {m.display_name} | Resource Name: {m.resource_name}")
        # Use the most recent trained model by default
        model = recent_models[0]
        print(f"\nUsing most recent model: {model.resource_name}")
    else:
        print("No models found in Vertex AI Model Registry.")
except Exception as e:
    print(f"Could not list models: {e}")

# Option 2: Alternatively set MODEL_RESOURCE_NAME in your .env or explicitly below:
if "model" not in locals() or not isinstance(model, aiplatform.Model):
    MODEL_RESOURCE_NAME = os.getenv("MODEL_RESOURCE_NAME", "")
    if MODEL_RESOURCE_NAME:
        model = aiplatform.Model(model_name=MODEL_RESOURCE_NAME)
        print(f"Loaded Vertex AI Model from environment: {model.resource_name}")
    else:
        print("Notice: Set MODEL_RESOURCE_NAME in your .env file to enable inference.")

## 1. Real-Time Online Inference

Deploy the model to a real-time endpoint (`n1-standard-4`), send prediction payloads, and undeploy the endpoint.

In [ ]:
# Sample JSON instance matching Bank Marketing feature columns
sample_instance = {
    "age": "35",
    "job": "technician",
    "marital": "married",
    "education": "tertiary",
    "default": "no",
    "balance": "1350",
    "housing": "yes",
    "loan": "no",
    "contact": "cellular",
    "day": "15",
    "month": "may",
    "duration": "220",
    "campaign": "1",
    "pdays": "-1",
    "previous": "0",
    "poutcome": "unknown",
}

print("Sample Feature Instance:", sample_instance)

# Uncomment below to deploy endpoint and run online predictions:
# if "model" in locals() and isinstance(model, aiplatform.Model):
#     print(f"Deploying model '{model.resource_name}' to endpoint...")
#     endpoint = deploy_model_to_endpoint(model=model, config=config)
#     predictions = predict_online(endpoint=endpoint, instances=[sample_instance])
#     print("Online Predictions Output:", predictions)
#     cleanup_endpoint(endpoint=endpoint)

## 2. Batch Inference

Submit a non-blocking batch prediction job for large offline datasets stored in Cloud Storage or BigQuery.

In [ ]:
gcs_input_csv = f"{config.bucket_uri}/test_instances.csv"
gcs_output_prefix = f"{config.root_dir}/batch_predictions"

print(f"Input GCS Source: {gcs_input_csv}")
print(f"Output GCS Destination Prefix: {gcs_output_prefix}")

# Uncomment below to submit batch prediction job to Vertex AI:
# if "model" in locals() and isinstance(model, aiplatform.Model):
#     print(f"Submitting Batch Prediction job for model '{model.resource_name}'...")
#     batch_job = run_batch_prediction(
#         model=model,
#         config=config,
#         gcs_source=gcs_input_csv,
#         gcs_destination_prefix=gcs_output_prefix,
#     )
#     print(f"Batch Prediction Job submitted successfully: {batch_job.resource_name}")